# Train the final model (LSTM bottleneck) on the exploration datasets

This notebook trains **the same model** selected in `03_train_stead_models`, `Branch_Encoder_Decoder_LSTM` with 178,442 parameters, on the three exploration datasets built by `09_build_exploration_mixtures`:

| Code | Category | Run folder |
|---|---|---|
| `VG2D` | 2D marine | `Branch_Encoder_Decoder_LSTM_vg2d` |
| `F3` | 3D marine | `Branch_Encoder_Decoder_LSTM_f3` |
| `STR3D` | 3D land | `Branch_Encoder_Decoder_LSTM_str3d` |

**The training is identical to the INSTANCE LSTM run of notebook 06.** It uses the same `step3_training_1D_presplit.py` (sha256 `d634fc8c…`), the same `config.py` (sha256 `146bcb7a…`; both hashes are checked in cell 3) and repo commit `afe2536`:
- `--presplit` (the split stored in the file, built by spatial block in notebook 09)
- batch 128, 100 epochs, float64, Adam 1e-3 with warm-up, no early stopping
- seed 99

Only the data file changes.

**How the notebook is organised**

| Cell | What it does |
|---|---|
| 1 | switches |
| 2 | setup: repo clone, lab log, file search |
| 3 | the training script and config, with hashes checked against the earlier runs |
| 4 | helper and metric code (the same as notebook 06) |
| 5 | data checks for each dataset |
| 6 | smoke run (2 epochs, 5,000 rows, LSTM) and the time projection |
| 7 | full training, one dataset after the other, with a time guard |
| 8 | quick check: each new model on its own test split, with the identity baseline, training curves and examples |

The full comparison with all other models is in notebook 11.

**Kaggle setup:**
- Accelerator: **GPU** (T4 or P100). Internet: **ON** (the repo is cloned).
- Attach the dataset **`wdn-exploration-mixtures`** (output of notebook 09).
- Run with **Save Version → Save & Run All (Commit)**.
- Expected time: about 45 min per dataset on a T4 (the INSTANCE LSTM with the same 18,000 / 6,000 rows took 42 min), so about 2.5 h for all three.
- Afterwards, make a dataset **`wdn-runs-exploration`** from the version output (the `wdn_runs/` folder).

**Resuming:** if a session stops early, attach `wdn-runs-exploration` (or the earlier version's output) and run again. Datasets that already have a finished 100-epoch run are skipped.

In [1]:
import time
T_SESSION0 = time.time()          # session clock for the time guard

# SWITCHES
RUN_SMOKE      = True             # 2-epoch LSTM run on 5,000 rows of the first dataset, projects the full time
RUN_FULL       = True             # full training, one dataset after the other
RUN_QUICK_EVAL = True             # each new model on its own test split (the full comparison is notebook 11)

TRAIN_DATASETS = ["VG2D", "F3", "STR3D"]
BOTTLENECK     = "LSTM"           # the final selected model; do not change
SEED           = 99
EPOCHS         = 100
SESSION_LIMIT_H = 12.0            # Kaggle GPU session limit
SAFETY_FACTOR  = 0.9
FORCE_FULL_RUN = False            # True = train without a smoke projection in this session

EPOCHS_REQUIRED = 100
EV_CLIP = -1.0
BOOT_SEED = 12345
EXAMPLE_SEED = 7
EXTRA_SEARCH_ROOTS = []
DATA_FILE = "training_datasets_{code}.hdf5"
CATEGORY = {"VG2D": "2D marine", "F3": "3D marine", "STR3D": "3D land"}
EXPECTED_SHA = {"config.py": "146bcb7aa227c74e", "step3_training_1D_presplit.py": "d634fc8c73dd624d"}   # as in the earlier runs
assert BOTTLENECK == "LSTM", "notebook 10 trains the final LSTM model only"

In [2]:
import os, sys, glob, json, time, shutil, subprocess, platform, datetime, re, traceback, hashlib
from pathlib import Path

ON_KAGGLE = os.path.exists("/kaggle/input")
BASE = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
RESULTS, LOGS, FIGS, RUNS_OUT = BASE/"results", BASE/"logs", BASE/"figures", BASE/"wdn_runs"
for d in (RESULTS, LOGS, FIGS, RUNS_OUT):
    d.mkdir(parents=True, exist_ok=True)
CODE_DIR = BASE/"WaveDecompNet-paper"
MODEL_DIR = BASE/"Model_and_datasets_1D_all_snr_40_unshuffled_equal_epoch100"
REPO_URL = "https://github.com/yinjiuxun/WaveDecompNet-paper"
EXPECTED_COMMIT = "afe2536cf29ead8b4d8d5c50ddf85cc1dd90ddd1"
SEARCH_ROOTS = (["/kaggle/input"] if ON_KAGGLE else []) + [str(p) for p in EXTRA_SEARCH_ROOTS]

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"{' '.join(map(str, cmd))}\n{r.stdout}\n{r.stderr}")
    return r.stdout.strip()

print("python", platform.python_version())
import numpy as np
print("numpy", np.__version__)
try:
    import torch
    print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0), "| memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
except Exception as e:
    print("torch import failed:", e)
print("disk free GB (working dir):", round(shutil.disk_usage(BASE).free / 1e9, 1))

def ensure(pip_name, module=None):
    try:
        __import__(module or pip_name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
ensure("h5py"); ensure("pandas"); ensure("scikit-learn", "sklearn"); ensure("scipy"); ensure("matplotlib")

if not (CODE_DIR/".git").exists():
    sh(["git", "clone", "--depth", "1", REPO_URL, str(CODE_DIR)])
GIT_HASH = sh(["git", "-C", str(CODE_DIR), "rev-parse", "HEAD"])
if GIT_HASH != EXPECTED_COMMIT:
    sh(["git", "-C", str(CODE_DIR), "fetch", "--depth", "1", "origin", EXPECTED_COMMIT])
    sh(["git", "-C", str(CODE_DIR), "checkout", EXPECTED_COMMIT])
    GIT_HASH = sh(["git", "-C", str(CODE_DIR), "rev-parse", "HEAD"])
assert GIT_HASH == EXPECTED_COMMIT, f"repo commit {GIT_HASH} differs from {EXPECTED_COMMIT}"
print("repo commit:", GIT_HASH)

def log_entry(part, command, outcome, seed=None):
    p = LOGS/"lab_log.md"
    if not p.exists():
        p.write_text("# Lab log\n\n| date (UTC) | part | command | commit | seed | outcome |\n|---|---|---|---|---|---|\n")
    now = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M")
    with open(p, "a") as f:
        f.write(f"| {now} | {part} | `{command}` | {GIT_HASH[:10]} | {seed if seed is not None else ''} | {outcome} |\n")

def find_files(name):
    out = []
    for r in SEARCH_ROOTS:
        out += glob.glob(f"{r}/**/{name}", recursive=True)
    return sorted(out)

def find_one(name, required=True):
    hits = find_files(name)
    if not hits and required:
        raise FileNotFoundError(f"{name} not found. Attach the dataset that contains it.")
    if len(hits) > 1:
        print(f"WARNING: {name} found {len(hits)} times, using {hits[0]}")
    return Path(hits[0]) if hits else None

python 3.13.16
numpy 2.1.3
torch 2.11.0+cu130 | cuda available: True
GPU: Tesla T4 | memory GB: 15.6
disk free GB (working dir): 20.9
repo commit: afe2536cf29ead8b4d8d5c50ddf85cc1dd90ddd1


In [3]:
CONFIG_PY = r'''"""Central configuration for the WaveDecompNet replication (works locally and on Kaggle).

Every other script imports this file, so paths and seeds live in exactly one place.
Override the STEAD location with the environment variable WDN_STEAD_DIR if needed.
"""
import glob
import os

ON_KAGGLE = os.path.exists("/kaggle/input")


def _find_stead(name):
    env = os.environ.get("WDN_STEAD_DIR")
    if env:
        return os.path.join(env, name)
    if ON_KAGGLE:
        hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
        if hits:
            return hits[0]
    return os.path.join(os.getcwd(), "stead", name)  # local fallback


# ---------------------------------------------------------------- paths
WORK_DIR = "/kaggle/working" if ON_KAGGLE else os.getcwd()

STEAD_HDF5 = _find_stead("merged.hdf5")
STEAD_CSV = _find_stead("merged.csv")

DATASET_DIR = os.path.join(WORK_DIR, "training_datasets")
POHA_DIR = os.path.join(WORK_DIR, "continuous_waveforms")
POHA_MSEED = os.path.join(POHA_DIR, "IU.POHA.00.20210601-20210801.mseed")

STEAD_MIX_H5 = os.path.join(DATASET_DIR, "training_datasets_STEAD_waveform_snr_40.hdf5")
POHA_MIX_H5 = os.path.join(DATASET_DIR, "training_datasets_STEAD_plus_POHA_snr_40_unshuffled.hdf5")
# paper-faithful name (the file is actually shuffled WITH replacement, see step2d)
FINAL_DATASET = os.path.join(DATASET_DIR, "training_datasets_all_snr_40_unshuffled.hdf5")
FINAL_DATASET_PERMUTED = os.path.join(DATASET_DIR, "training_datasets_all_snr_40_permuted.hdf5")

MODEL_DIR = os.path.join(WORK_DIR, "Model_and_datasets_1D_all_snr_40_unshuffled_equal_epoch100")

# ---------------------------------------------------------------- data recipe (from the repo)
SNR_MIN_DB = 40
N_EVENTS = 100000                 # step2b: number of STEAD mixtures
F_DOWNSAMPLE = 10                 # Hz
N_APPEND = 1000                   # chunk size when writing HDF5
STEAD_SEEDS = dict(earthquake=101, noise=102, shift=103, snr=104)   # step2b
POHA_SEEDS = dict(earthquake=111, noise=112, shift=113, snr=114)    # step2c
POHA_AMPLITUDE_THRESHOLD = 6      # step2c: drop samples above 6 x median amplitude
POHA_EXPECTED_FS = 40.0           # step2c calls decimate(4) -> 10 Hz, so it assumes 40 Hz

# step2c in the original code writes only full chunks of N_APPEND, so the last
# (N mod 1000) windows are silently dropped. False = keep that behaviour (fidelity).
KEEP_REMAINDER = False

# step2d in the original code shuffles with np.random.choice(N, N): WITH replacement,
# no seed. True = same behaviour but seeded and logged; False = a proper permutation.
SHUFFLE_WITH_REPLACEMENT = True
SHUFFLE_SEED = 0

# ---------------------------------------------------------------- training (step3)
BATCH_SIZE = 128
EPOCHS = 100
LR = 1e-3
MIN_EPOCHS = 30
PATIENCE = None                   # None = no early stopping (always 100 epochs)
TORCH_SEED = 99
SPLIT_SEED_1 = 13
SPLIT_SEED_2 = 20
TRAIN_SIZE = 0.6
TEST_SIZE = 0.5                   # of the remaining 40 % -> 20 % val, 20 % test
'''

STEP3_PRESPLIT_PY = r'''"""step3_presplit = step3 (patched) + ONE new option: --presplit.

--presplit uses the dataset split stored in the data file (dataset `split_id`: 0 = train, 1 = validation, 2 = test) instead of the
random 60/20/20 split after loading. Needed for the INSTANCE data, which is split by earthquake BEFORE mixing. Without --presplit the
behaviour is identical to step3_training_1D.py.

Original docstring:
step3 (patched): train ONE bottleneck per run (needed for Kaggle's session limits).

Same model, loss, optimiser, LR schedule, split (60/20/20, seeds 13 and 20) and seeds as the repo.
Changes: command-line arguments, paths from config.py, no `tabnanny` import, no `verbose=` in
LambdaLR (removed in newer PyTorch), figure saved headless, run info saved as JSON.

Quick smoke test first (a few minutes):
    python step3_training_1D.py --bottleneck None --epochs 2 --subset 5000 --tag _smoke
Full run:
    python step3_training_1D.py --bottleneck LSTM
"""
import argparse
import copy
import json
import os
import random
import time

import h5py
import matplotlib
matplotlib.use("Agg")
import numpy as np
import torch
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

import config as C
from autoencoder_1D_models_torch import SeismogramDecoder, SeismogramEncoder, SeisSeparator
from torch_tools import WaveformDataset, training_loop_branches, try_gpu

BOTTLENECKS = ["None", "Linear", "LSTM", "attention", "Transformer"]


def make_bottleneck(name):
    if name == "None":
        return None
    if name == "Linear":
        return torch.nn.Linear(64, 64, dtype=torch.float64)
    if name == "LSTM":
        return torch.nn.LSTM(64, 32, 2, bidirectional=True, batch_first=True, dtype=torch.float64)
    if name == "attention":
        return torch.nn.MultiheadAttention(64, 4, 0.2, batch_first=True, dtype=torch.float64)
    if name == "Transformer":
        layer = torch.nn.TransformerEncoderLayer(d_model=64, nhead=4, dtype=torch.float64)
        return torch.nn.TransformerEncoder(layer, num_layers=1)
    raise NameError("bottleneck type has not been defined!")


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--bottleneck", required=True, choices=BOTTLENECKS)
    p.add_argument("--data", default=C.FINAL_DATASET)
    p.add_argument("--seed", type=int, default=C.TORCH_SEED)
    p.add_argument("--epochs", type=int, default=C.EPOCHS)
    p.add_argument("--subset", type=int, default=None, help="use only the first N samples (smoke test)")
    p.add_argument("--tag", default=None, help="suffix of the output folder")
    p.add_argument("--presplit", action="store_true", help="use split_id stored in the data file (0 train, 1 val, 2 test)")
    args = p.parse_args()

    tag = args.tag if args.tag is not None else ("_warmup" if args.seed == C.TORCH_SEED else f"_warmup_seed{args.seed}")
    model_structure = "Branch_Encoder_Decoder"
    model_name = f"{model_structure}_{args.bottleneck}"
    out_dir = os.path.join(C.MODEL_DIR, model_name + tag)
    os.makedirs(out_dir, exist_ok=True)
    device = try_gpu()
    print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

    # ---- data
    print("#" * 12 + " Loading data " + "#" * 12)
    if args.presplit:
        with h5py.File(args.data, "r") as f:
            n_all = f["X_train"].shape[0]
            assert "split_id" in f, "--presplit needs a split_id dataset in the data file"
            sid_all = f["split_id"][:]
            if args.subset is None or args.subset >= n_all:
                sel = np.arange(n_all)
                X_train, Y_train = f["X_train"][:], f["Y_train"][:]
            else:                                    # evenly spaced rows keep the train / val / test proportions
                sel = np.unique(np.linspace(0, n_all - 1, args.subset).astype(int))
                X_train, Y_train = f["X_train"][sel], f["Y_train"][sel]
            sid = sid_all[sel]
        print("data:", X_train.shape, X_train.dtype, "| presplit rows train/val/test:", [int((sid == k).sum()) for k in range(3)])
        assert (sid == 0).any() and (sid == 1).any(), "no train or validation rows selected"
        X_training, Y_training = X_train[sid == 0], Y_train[sid == 0]
        X_validate, Y_validate = X_train[sid == 1], Y_train[sid == 1]
        del X_train, Y_train
    else:
        with h5py.File(args.data, "r") as f:
            n = f["X_train"].shape[0] if args.subset is None else min(args.subset, f["X_train"].shape[0])
            X_train, Y_train = f["X_train"][:n], f["Y_train"][:n]
        print("data:", X_train.shape, X_train.dtype)

        X_training, X_test, Y_training, Y_test = train_test_split(
            X_train, Y_train, train_size=C.TRAIN_SIZE, random_state=C.SPLIT_SEED_1)
        X_validate, X_test, Y_validate, Y_test = train_test_split(
            X_test, Y_test, test_size=C.TEST_SIZE, random_state=C.SPLIT_SEED_2)
        del X_train, Y_train, X_test, Y_test

    # ---- seeds (same as the repo, torch seed configurable for multi-seed runs)
    torch.manual_seed(args.seed)
    random.seed(0)
    np.random.seed(20)
    torch.backends.cudnn.benchmark = False

    training_data = WaveformDataset(X_training, Y_training)
    validate_data = WaveformDataset(X_validate, Y_validate)

    # ---- model
    bottleneck = make_bottleneck(args.bottleneck)
    encoder = SeismogramEncoder()
    decoder_earthquake = SeismogramDecoder(bottleneck=copy.deepcopy(bottleneck))
    decoder_noise = SeismogramDecoder(bottleneck=copy.deepcopy(bottleneck))
    model = SeisSeparator(model_name, encoder, decoder_earthquake, decoder_noise).to(device=device)
    n_params = int(sum(p.numel() for p in model.parameters()))
    print(f"{model_name}: {n_params} parameters")

    batch_size, lr = C.BATCH_SIZE, C.LR
    loss_fn = torch.nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    LR_func = lambda epoch: (epoch + 1) / 11 if (epoch <= 10) else (0.95 ** epoch if (epoch < 50) else 0.95 ** 50)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=LR_func)

    train_iter = DataLoader(training_data, batch_size=batch_size, shuffle=True)
    validate_iter = DataLoader(validate_data, batch_size=batch_size, shuffle=False)

    # ---- train
    print("#" * 12 + " training model " + model_name + " " + "#" * 12)
    t0 = time.time()
    model, avg_train_losses, avg_valid_losses, partial_loss = training_loop_branches(
        train_iter, validate_iter, model, loss_fn, optimizer, scheduler,
        epochs=args.epochs, patience=C.PATIENCE, device=device, minimum_epochs=C.MIN_EPOCHS)
    runtime = time.time() - t0
    print(f"Training is done! {runtime / 60:.1f} min")

    # ---- save
    torch.save(model, os.path.join(out_dir, f"{model_name}_Model.pth"))
    with h5py.File(os.path.join(out_dir, f"{model_name}_Training_history.hdf5"), "w") as f:
        f.create_dataset("loss", data=avg_train_losses)
        f.create_dataset("val_loss", data=avg_valid_losses)
        f.create_dataset("earthquake_loss", data=partial_loss[0])
        f.create_dataset("earthquake_val_loss", data=partial_loss[1])
        f.create_dataset("noise_loss", data=partial_loss[2])
        f.create_dataset("noise_val_loss", data=partial_loss[3])
    with h5py.File(os.path.join(out_dir, f"{model_name}_Dataset_split.hdf5"), "w") as f:
        f.attrs["model_name"] = model_name
        f.attrs["train_size"] = C.TRAIN_SIZE
        f.attrs["test_size"] = C.TEST_SIZE
        f.attrs["rand_seed1"] = C.SPLIT_SEED_1
        f.attrs["rand_seed2"] = C.SPLIT_SEED_2
        f.attrs["presplit"] = bool(args.presplit)
    with open(os.path.join(out_dir, "run_info.json"), "w") as f:
        json.dump(dict(model_name=model_name, bottleneck=args.bottleneck, torch_seed=args.seed,
                       epochs=args.epochs, subset=args.subset, data=args.data, presplit=bool(args.presplit),
                       n_train=len(training_data), n_val=len(validate_data), n_params=n_params,
                       runtime_s=runtime, device=str(device),
                       gpu=torch.cuda.get_device_name(0) if device.type == "cuda" else None,
                       torch=torch.__version__, final_train_loss=float(avg_train_losses[-1]),
                       final_val_loss=float(avg_valid_losses[-1])), f, indent=2)

    plt.figure()
    plt.plot(avg_train_losses, "o", label="loss")
    plt.plot(avg_valid_losses, "-", label="Validation loss")
    for k, (lab, mk) in enumerate(zip(["earthquake train loss", "earthquake valid loss",
                                       "noise train loss", "noise valid loss"], ["o", "", "o", ""])):
        plt.plot(partial_loss[k], marker=mk, label=lab)
    plt.yscale("log")
    plt.legend()
    plt.title(model_name)
    plt.savefig(os.path.join(out_dir, f"{model_name}_Training_history.png"))
    plt.close()
    print("saved to", out_dir)


if __name__ == "__main__":
    main()
'''

(CODE_DIR/"config.py").write_text(CONFIG_PY)
(CODE_DIR/"step3_training_1D_presplit.py").write_text(STEP3_PRESPLIT_PY)
SCRIPT_SHA = {n: hashlib.sha256((CODE_DIR/n).read_bytes()).hexdigest()[:16] for n in ("config.py", "step3_training_1D_presplit.py")}
print("scripts written, sha256[:16]:", SCRIPT_SHA)
assert SCRIPT_SHA == EXPECTED_SHA, f"training scripts differ from the earlier runs: {SCRIPT_SHA} vs {EXPECTED_SHA}"
# where step3 writes its runs: config.WORK_DIR = /kaggle/working on Kaggle, else the script's working directory (the repo folder)
MODEL_DIR = (Path("/kaggle/working") if ON_KAGGLE else CODE_DIR)/"Model_and_datasets_1D_all_snr_40_unshuffled_equal_epoch100"
print("identical to the scripts of the STEAD / INSTANCE / variant runs: OK")
tt = CODE_DIR/"torch_tools.py"; txt = tt.read_text()
if "np.Inf" in txt:
    tt.write_text(txt.replace("np.Inf", "np.inf")); print("patched np.Inf -> np.inf in torch_tools.py (same patch as notebooks 03 / 06)")
assert "np.Inf" not in tt.read_text()
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

scripts written, sha256[:16]: {'config.py': '146bcb7aa227c74e', 'step3_training_1D_presplit.py': 'd634fc8c73dd624d'}
identical to the scripts of the STEAD / INSTANCE / variant runs: OK
patched np.Inf -> np.inf in torch_tools.py (same patch as notebooks 03 / 06)


In [4]:
import pandas as pd
import h5py

import pandas as pd

def read_csv_keep(path):
    """pandas treats the string 'None' (our bottleneck name) as missing by default; keep it."""
    return pd.read_csv(path, keep_default_na=False, na_values=[""])

def save_fig(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(FIGS/f"{name}.{ext}", bbox_inches="tight", dpi=200)

def prepare_torch():
    """Import torch and the repo modules; disable the eval fast path so eval == the training-time math."""
    import importlib, torch
    if str(CODE_DIR) not in sys.path:
        sys.path.insert(0, str(CODE_DIR))
    mods = {n: importlib.import_module(n) for n in ("autoencoder_1D_models_torch", "torch_tools", "utilities")}
    try:
        torch.backends.mha.set_fastpath_enabled(False)
    except Exception:
        pass
    return torch, mods

COMPAT_DEFAULTS = {"norm_first": False, "activation_relu_or_gelu": 1, "enable_nested_tensor": False,
                   "use_nested_tensor": False, "mask_check": True}
COMPAT_LOG = []

def load_model(path, device):
    """Load a whole pickled model (torch.save(model)). Needs autoencoder_1D_models_torch importable."""
    torch, _ = prepare_torch()
    try:
        m = torch.load(str(path), map_location=device, weights_only=False)
    except TypeError:
        m = torch.load(str(path), map_location=device)
    return m.to(device).double().eval()

def check_forward(model, device):
    """Dummy forward pass. Optional compat shim ONLY if an old pickle lacks an attribute newer torch expects
    (defaults = what the constructor sets). Every use is recorded in COMPAT_LOG and goes into the deviation log."""
    torch, _ = prepare_torch()
    x = torch.randn(2, 3, 600, dtype=torch.float64, device=device)
    for _ in range(8):
        try:
            with torch.no_grad():
                o = model(x)
            assert len(o) == 2 and tuple(o[0].shape) == (2, 3, 600) and tuple(o[1].shape) == (2, 3, 600), "unexpected output shapes"
            return True, None
        except AttributeError as e:
            name = (re.search(r"has no attribute '(\w+)'", str(e)) or [None, None])[1]
            if name not in COMPAT_DEFAULTS:
                return False, traceback.format_exc()
            for mod in model.modules():
                if type(mod).__name__ in ("TransformerEncoderLayer", "TransformerEncoder") and not hasattr(mod, name):
                    setattr(mod, name, COMPAT_DEFAULTS[name])
            COMPAT_LOG.append(name)
        except Exception:
            return False, traceback.format_exc()
    return False, "compat shim did not converge: " + ", ".join(COMPAT_LOG)

def predict(model, Xt, device, bs=256):
    """Xt: (N, 3, 600) float64 -> (earthquake, noise), both (N, 3, 600)."""
    torch, _ = prepare_torch()
    pe = np.empty_like(Xt); pn = np.empty_like(Xt)
    with torch.no_grad():
        for i in range(0, len(Xt), bs):
            xb = torch.from_numpy(Xt[i:i + bs]).to(device)
            o1, o2 = model(xb)
            pe[i:i + bs] = o1.cpu().numpy(); pn[i:i + bs] = o2.cpu().numpy()
    return pe, pn

def get_device():
    torch, _ = prepare_torch()
    return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")



def ev_score(y, p, axis=-1):
    """Explained variance exactly as sklearn.explained_variance_score(multioutput='raw_values') (the definition used in step5),
    vectorised: one value per trace and component, time on `axis`."""
    d = y - p
    num = np.mean((d - d.mean(axis, keepdims=True)) ** 2, axis=axis)
    den = np.mean((y - y.mean(axis, keepdims=True)) ** 2, axis=axis)
    out = np.ones_like(num)
    ok = (num != 0) & (den != 0)
    out[ok] = 1 - num[ok] / den[ok]
    out[(num != 0) & (den == 0)] = 0.0
    return out

def step5_snr_db(signal, noise, axis=-1):
    """signal_to_noise_ratio() of step5, in dB, per component."""
    s0 = np.sum(signal ** 2, axis=axis) / (np.sum(noise ** 2, axis=axis) + 1e-6)
    return 10 * np.log10(s0 + 1e-6)

def pearson_rows(a, b):
    a = a.reshape(len(a), -1); b = b.reshape(len(b), -1)
    a = a - a.mean(1, keepdims=True); b = b - b.mean(1, keepdims=True)
    den = np.sqrt((a ** 2).sum(1) * (b ** 2).sum(1))
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(den > 0, (a * b).sum(1) / den, np.nan)

def metrics_chunked(Xt, Yt, pe, pn, chunk=4000):
    """Per-sample metrics. Xt, Yt, pe, pn: (N, 3, 600). Earthquake truth = Yt, noise truth = Xt - Yt (as in training)."""
    cols = {k: [] for k in ("ev_eq", "ev_nz", "snr5", "mse_eq", "mse_noise", "corr")}
    for i in range(0, len(Xt), chunk):
        x, y, a, b = Xt[i:i + chunk], Yt[i:i + chunk], pe[i:i + chunk], pn[i:i + chunk]
        tn = x - y
        cols["ev_eq"].append(ev_score(y, a)); cols["ev_nz"].append(ev_score(tn, b))
        cols["snr5"].append(step5_snr_db(y, tn))
        cols["mse_eq"].append(((y - a) ** 2).mean((1, 2))); cols["mse_noise"].append(((tn - b) ** 2).mean((1, 2)))
        cols["corr"].append(pearson_rows(y, a))
    return {k: np.concatenate(v) for k, v in cols.items()}

def boot_ci_mean(x, n_boot=1000, rng=None, alpha=0.05, chunk=100):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan, np.nan
    rng = rng or np.random.default_rng(BOOT_SEED)
    means = np.concatenate([x[rng.integers(0, len(x), (min(chunk, n_boot - i), len(x)))].mean(1) for i in range(0, n_boot, chunk)])
    return float(x.mean()), float(np.quantile(means, alpha / 2)), float(np.quantile(means, 1 - alpha / 2))

def boot_ci_median(x, n_boot=1000, rng=None, alpha=0.05, chunk=100):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    rng = rng or np.random.default_rng(BOOT_SEED)
    med = np.concatenate([np.median(x[rng.integers(0, len(x), (min(chunk, n_boot - i), len(x)))], axis=1) for i in range(0, n_boot, chunk)])
    return float(np.median(x)), float(np.quantile(med, alpha / 2)), float(np.quantile(med, 1 - alpha / 2))

def snr_edges(lo=0.1, hi=100.0, per_decade=4):
    return np.logspace(np.log10(lo), np.log10(hi), int(round(np.log10(hi / lo) * per_decade)) + 1)

def bin_index(snr, edges):
    b = np.searchsorted(edges, snr, side="right") - 1
    b[snr == edges[-1]] = len(edges) - 2
    b[(snr < edges[0]) | (snr > edges[-1]) | ~np.isfinite(snr)] = -1
    return b

def summarise_model(df):
    """Model-level numbers. Pooled medians follow step5 (every trace-component is one value)."""
    n5 = (len(df) // 256) * 256            # step5 drops the last incomplete batch of 256
    def pool(d, p): return d[[f"{p}_c0", f"{p}_c1", f"{p}_c2"]].to_numpy().ravel()
    e, nz = pool(df, "ev_eq"), pool(df, "ev_noise"); d5 = df.iloc[:n5]
    return dict(n_samples=len(df),
                median_ev_eq=float(np.median(e)), median_ev_noise=float(np.median(nz)),
                mean_ev_eq_clipped=float(np.clip(e, EV_CLIP, 1).mean()), mean_ev_noise_clipped=float(np.clip(nz, EV_CLIP, 1).mean()),
                frac_ev_eq_lt0=float((e < 0).mean()), frac_ev_noise_lt0=float((nz < 0).mean()),
                test_loss=float(df["mse_total"].mean()), mean_mse_eq=float(df["mse_eq"].mean()), mean_mse_noise=float(df["mse_noise"].mean()),
                mean_corr_eq=float(np.nanmean(df["corr_eq"])),
                median_ev_eq_step5subset=float(np.median(pool(d5, "ev_eq"))), median_ev_noise_step5subset=float(np.median(pool(d5, "ev_noise"))),
                test_loss_step5subset=float(d5["mse_total"].mean()))

In [5]:
DATA = {}
for code_ in TRAIN_DATASETS:
    path = find_one(DATA_FILE.format(code=code_), required=False)
    if path is None:
        print(f"{code_}: {DATA_FILE.format(code=code_)} NOT FOUND. Attach wdn-exploration-mixtures (output of notebook 09)."); continue
    with h5py.File(path, "r") as f:
        X = f["X_train"]; sid = f["split_id"][:]
        assert X.shape == f["Y_train"].shape and X.shape[1:] == (600, 3) and X.dtype == np.float64, f"{code_}: unexpected shape/dtype {X.shape} {X.dtype}"
        assert (np.diff(sid.astype(int)) >= 0).all(), f"{code_}: rows must be ordered train | val | test"
        attrs = {k: (v.item() if hasattr(v, "item") else v) for k, v in f.attrs.items()}
        DATA[code_] = dict(path=path, n=int(X.shape[0]), n_train=int((sid == 0).sum()), n_val=int((sid == 1).sum()), n_test=int((sid == 2).sum()),
                           dt_s=float(attrs.get("dt_s", np.nan)), noise_mode=str(attrs.get("noise_mode", "")),
                           signal=("brute stack" if attrs.get("signal_is_brute_stack") else str(attrs.get("signal_file", ""))))
    rep_p = find_files(f"build_report_{code_}.json")
    DATA[code_]["build_report"] = rep_p[0] if rep_p else None
    print(f"{code_:6s} {CATEGORY[code_]:10s} rows {DATA[code_]['n']} (train {DATA[code_]['n_train']}, val {DATA[code_]['n_val']}, test {DATA[code_]['n_test']}) | "
          f"dt {DATA[code_]['dt_s'] * 1e3:.3g} ms | signal: {DATA[code_]['signal']} | noise: {DATA[code_]['noise_mode']}\n       {path}")
json.dump({k: {**v, "path": str(v["path"])} for k, v in DATA.items()}, open(RESULTS/"exploration_data_facts.json", "w"), indent=2, default=str)
assert DATA, "no exploration dataset found: attach wdn-exploration-mixtures"
gpu_ok = False
try:
    import torch; gpu_ok = torch.cuda.is_available()
except Exception:
    pass
print("\nGPU visible:", gpu_ok, "" if gpu_ok else "<- turn on a GPU accelerator before training")

VG2D   2D marine  rows 30000 (train 18000, val 6000, test 6000) | dt 4 ms | signal: brute stack | noise: prefirstbreak
       /kaggle/input/datasets/prabhanshu22/wdn-exploration-mixtures/training_datasets_VG2D.hdf5
F3     3D marine  rows 30000 (train 18000, val 6000, test 6000) | dt 1.33 ms | signal: seismic_entire_volume.npy | noise: synthetic
       /kaggle/input/datasets/prabhanshu22/wdn-exploration-mixtures/training_datasets_F3.hdf5
STR3D  3D land    rows 30000 (train 18000, val 6000, test 6000) | dt 2 ms | signal: Stratton3D_32bit.sgy | noise: synthetic
       /kaggle/input/datasets/prabhanshu22/wdn-exploration-mixtures/training_datasets_STR3D.hdf5

GPU visible: True 


## Training

In [6]:
def run_tag(code_, seed=SEED):
    return f"_{code_.lower()}" if seed == 99 else f"_{code_.lower()}_seed{seed}"

def run_step3(bottleneck, seed, epochs, data, subset=None, tag=None, log_name="train.log"):
    cmd = [sys.executable, "-u", "step3_training_1D_presplit.py", "--presplit", "--bottleneck", bottleneck, "--data", str(data),
           "--seed", str(seed), "--epochs", str(epochs)]
    if subset: cmd += ["--subset", str(subset)]
    if tag is not None: cmd += ["--tag", tag]
    print(" ".join(cmd)); t0 = time.time(); lines = []
    proc = subprocess.Popen(cmd, cwd=CODE_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    with open(LOGS/log_name, "w") as lf:
        for line in proc.stdout:
            print(line, end=""); lf.write(line); lines.append(line)
    return proc.wait(), "".join(lines), time.time() - t0

def finished_run(code_):
    """A finished 100-epoch run for this dataset, in wdn_runs/ of this session or in an attached dataset."""
    folder = f"Branch_Encoder_Decoder_{BOTTLENECK}{run_tag(code_)}"
    for root in [str(RUNS_OUT)] + SEARCH_ROOTS:
        for p in glob.glob(f"{root}/**/{folder}/run_info.json", recursive=True):
            try: info = json.load(open(p))
            except Exception: continue
            if info.get("epochs") == EPOCHS_REQUIRED and info.get("subset") is None and Path(p).with_name(f"{info['model_name']}_Model.pth").exists():
                return Path(p).parent, info
    return None, None

def gpu_warning():
    try:
        import torch
        if not torch.cuda.is_available():
            print("WARNING: no GPU visible. Turn on a GPU accelerator in the notebook settings."); return
        n = torch.cuda.get_device_name(0); print("GPU:", n)
        if "T4" in n:
            print("NOTE: T4 has slow float64 (this model is float64). Read the smoke projection before a full run.")
    except Exception:
        pass

TODO = [c for c in TRAIN_DATASETS if c in DATA and finished_run(c)[0] is None]
print("already finished:", [c for c in TRAIN_DATASETS if c in DATA and c not in TODO], "| to train:", TODO)

SMOKE = {}
if RUN_SMOKE and TODO:
    gpu_warning()
    c0 = TODO[0]
    rc, out, dt = run_step3(BOTTLENECK, SEED, 2, DATA[c0]["path"], subset=5000, tag=f"_{c0.lower()}_smoke", log_name=f"smoke_{c0}.log")
    assert rc == 0, "smoke run failed, read the traceback above"
    times = [float(x) for x in re.findall(r"time per epoch: ([\d.]+) s", out)]
    info = json.load(open(MODEL_DIR/f"Branch_Encoder_Decoder_{BOTTLENECK}_{c0.lower()}_smoke"/"run_info.json"))
    per_row = times[-1] / (info["n_train"] + info["n_val"])
    proj = {c: per_row * (DATA[c]["n_train"] + DATA[c]["n_val"]) * EPOCHS / 3600 for c in TODO}
    total = sum(proj.values()); left = SESSION_LIMIT_H - (time.time() - T_SESSION0) / 3600
    SMOKE.update(epoch_times=times, per_row_s=per_row, projected_hours=proj, total_hours=total, hours_left=left, ok=total <= SAFETY_FACTOR * left)
    print(f"\nsmoke ({BOTTLENECK}, {c0}) epoch times (s): {times}")
    print("PROJECTED hours per dataset:", {k: round(v, 2) for k, v in proj.items()}, f"| total {total:.2f} h | session time left {left:.2f} h")
    print("OK to train all of them in this session" if SMOKE["ok"] else
          "NOTE: not all fit in this session. The time guard trains as many as fit; run the rest in a new session (finished runs are skipped).")
    json.dump(SMOKE, open(RESULTS/"smoke_projection_exploration.json", "w"), indent=2)
    log_entry("10 smoke", f"step3_presplit --bottleneck {BOTTLENECK} --epochs 2 --subset 5000 ({c0})", f"epoch times {times}, projected total {total:.2f} h", SEED)
else:
    print("smoke run skipped" + ("" if TODO else " (nothing left to train)"))

already finished: [] | to train: ['VG2D', 'F3', 'STR3D']
GPU: Tesla T4
NOTE: T4 has slow float64 (this model is float64). Read the smoke projection before a full run.
/usr/bin/python3 -u step3_training_1D_presplit.py --presplit --bottleneck LSTM --data /kaggle/input/datasets/prabhanshu22/wdn-exploration-mixtures/training_datasets_VG2D.hdf5 --seed 99 --epochs 2 --subset 5000 --tag _vg2d_smoke
device: cuda:0 Tesla T4
############ Loading data ############
data: (5000, 600, 3) float64 | presplit rows train/val/test: [3000, 1000, 1000]
Branch_Encoder_Decoder_LSTM: 178442 parameters
############ training model Branch_Encoder_Decoder_LSTM ############
[1/2] train_loss: 2.73609 valid_loss: 1.23349
time per epoch: 7.554 s
[2/2] train_loss: 2.26979 valid_loss: 1.71989
time per epoch: 4.141 s
Training is done! 0.2 min
saved to /kaggle/working/Model_and_datasets_1D_all_snr_40_unshuffled_equal_epoch100/Branch_Encoder_Decoder_LSTM_vg2d_smoke

smoke (LSTM, VG2D) epoch times (s): [7.554, 4.141]
PROJE

In [7]:
TRAINED = {}
if RUN_FULL and TODO:
    if not SMOKE and not FORCE_FULL_RUN:
        raise RuntimeError("Run the smoke test first in this session (RUN_SMOKE) or set FORCE_FULL_RUN = True.")
    gpu_warning()
    for c in TODO:
        left = SESSION_LIMIT_H - (time.time() - T_SESSION0) / 3600
        need = SMOKE.get("projected_hours", {}).get(c)
        if need is not None and need > SAFETY_FACTOR * left:
            print(f"\nSTOP before {c}: needs about {need:.2f} h, only {left:.2f} h left in the session. "
                  "Save this version, attach its output and run again: finished datasets are skipped."); break
        print("\n" + "=" * 100 + f"\n{c} ({CATEGORY[c]}): full training, {BOTTLENECK}, seed {SEED}, {EPOCHS} epochs")
        rc, out, dt = run_step3(BOTTLENECK, SEED, EPOCHS, DATA[c]["path"], tag=run_tag(c), log_name=f"train_{c}_{BOTTLENECK}_seed{SEED}.log")
        if rc != 0:
            print(f"{c}: TRAINING FAILED (return code {rc}), read the traceback above; continuing with the next dataset"); continue
        folder = f"Branch_Encoder_Decoder_{BOTTLENECK}{run_tag(c)}"; src_dir = MODEL_DIR/folder
        info = json.load(open(src_dir/"run_info.json"))
        shutil.copytree(src_dir, RUNS_OUT/folder, dirs_exist_ok=True)
        rep_sha = hashlib.sha256(open(DATA[c]["build_report"], "rb").read()).hexdigest()[:16] if DATA[c]["build_report"] else None
        cfg = dict(git_hash=GIT_HASH, seed=SEED, bottleneck=BOTTLENECK, trained_on=c, category=CATEGORY[c], dataset=str(info.get("data")), presplit=True,
                   build_report_sha=rep_sha, torch=info.get("torch"), gpu=info.get("gpu"), numpy=np.__version__, python=platform.python_version(),
                   script_sha=SCRIPT_SHA, date_utc=datetime.datetime.now(datetime.timezone.utc).isoformat())
        (RUNS_OUT/folder/f"config_{info['model_name']}.json").write_text(json.dumps(cfg, indent=2))
        TRAINED[c] = info
        print(f"\nDONE {folder}: {info['runtime_s']/60:.1f} min, {info['n_params']} params, final train loss {info['final_train_loss']:.5f}, "
              f"final val loss {info['final_val_loss']:.5f}, n_train {info['n_train']}, n_val {info['n_val']}")
        log_entry("10 full", f"step3_presplit --bottleneck {BOTTLENECK} --seed {SEED} --epochs {EPOCHS} ({c})",
                  f"{info['runtime_s']/60:.1f} min, val loss {info['final_val_loss']:.5f}", SEED)
    print("\ntrained in this session:", list(TRAINED), "| run folders are in", RUNS_OUT, "-> make the dataset wdn-runs-exploration from the version output")
else:
    print("full training skipped" + ("" if TODO else " (nothing left to train)"))

GPU: Tesla T4
NOTE: T4 has slow float64 (this model is float64). Read the smoke projection before a full run.

VG2D (2D marine): full training, LSTM, seed 99, 100 epochs
/usr/bin/python3 -u step3_training_1D_presplit.py --presplit --bottleneck LSTM --data /kaggle/input/datasets/prabhanshu22/wdn-exploration-mixtures/training_datasets_VG2D.hdf5 --seed 99 --epochs 100 --tag _vg2d
device: cuda:0 Tesla T4
############ Loading data ############
data: (30000, 600, 3) float64 | presplit rows train/val/test: [18000, 6000, 6000]
Branch_Encoder_Decoder_LSTM: 178442 parameters
############ training model Branch_Encoder_Decoder_LSTM ############
[  1/100] train_loss: 2.07932 valid_loss: 1.54391
time per epoch: 25.137 s
[  2/100] train_loss: 1.08486 valid_loss: 0.87800
time per epoch: 24.814 s
[  3/100] train_loss: 0.69903 valid_loss: 0.63964
time per epoch: 25.108 s
[  4/100] train_loss: 0.54379 valid_loss: 0.48681
time per epoch: 25.477 s
[  5/100] train_loss: 0.43458 valid_loss: 0.38451
time per 

## Quick check on each dataset's own test split

This is a sanity check only. The comparison with the STEAD, INSTANCE, variant and pretrained models, the cross-dataset matrix and the figures for the report are in notebook 11.

In [8]:
QUICK = []
if RUN_QUICK_EVAL:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    torch, mods = prepare_torch(); dev = get_device()
    runs = {c: finished_run(c) for c in TRAIN_DATASETS if c in DATA}
    runs = {c: r for c, r in runs.items() if r[0] is not None}
    if not runs:
        print("no finished run to check")
    # training curves
    if runs:
        fig, ax = plt.subplots(1, len(runs), figsize=(5 * len(runs), 3.6), squeeze=False)
        for a, (c, (d, info)) in zip(ax[0], runs.items()):
            with h5py.File(d/f"{info['model_name']}_Training_history.hdf5", "r") as f:
                a.plot(f["loss"][:], label="train"); a.plot(f["val_loss"][:], label="validation")
            a.set_yscale("log"); a.set_xlabel("epoch"); a.set_title(f"{c} ({CATEGORY[c]})"); a.grid(alpha=.3)
        ax[0, 0].set_ylabel("MSE loss (earthquake/signal + noise)"); ax[0, 0].legend(); plt.tight_layout(); save_fig(fig, "fig_exploration_training_curves"); plt.show()
    for c, (d, info) in runs.items():
        with h5py.File(DATA[c]["path"], "r") as f:
            sid = f["split_id"][:]; te = np.where(sid == 2)[0]
            Xte = f["X_train"][te[0]:te[-1] + 1]; Yte = f["Y_train"][te[0]:te[-1] + 1]; tt_ = f["time"][:]
        Xt = np.ascontiguousarray(np.moveaxis(Xte, 1, -1)); Yt = np.ascontiguousarray(np.moveaxis(Yte, 1, -1)); del Xte, Yte
        model = load_model(d/f"{info['model_name']}_Model.pth", dev); ok, err = check_forward(model, dev); assert ok, err
        pe, pn = predict(model, Xt, dev)
        for mid, a_, b_ in ((f"{c.lower()}_{BOTTLENECK}_seed{SEED}", pe, pn), ("baseline_identity", Xt, Xt)):
            m = metrics_chunked(Xt, Yt, a_, b_)
            QUICK.append(dict(dataset=c, category=CATEGORY[c], model_id=mid, n_test=len(Xt),
                              median_ev_signal=float(np.median(m["ev_eq"])), median_ev_noise=float(np.median(m["ev_nz"])),
                              mean_ev_signal_clipped=float(np.clip(m["ev_eq"], EV_CLIP, 1).mean()), mean_ev_noise_clipped=float(np.clip(m["ev_nz"], EV_CLIP, 1).mean()),
                              test_loss=float((m["mse_eq"] + m["mse_noise"]).mean()), mean_corr_signal=float(np.nanmean(m["corr"])),
                              final_val_loss=info.get("final_val_loss") if "baseline" not in mid else None, runtime_min=info.get("runtime_s", 0) / 60 if "baseline" not in mid else None))
            if "baseline" not in mid: v = np.clip(m["ev_eq"], EV_CLIP, 1).mean(1)
        # examples: best 10 %, median, worst 2 %, channel 0
        rng = np.random.default_rng(EXAMPLE_SEED); order_ = np.argsort(v)
        pick = {"good": rng.choice(order_[-max(1, len(v) // 10):]), "median": order_[len(v) // 2], "failure": rng.choice(order_[:max(1, len(v) // 50)])}
        fig, ax = plt.subplots(3, 3, figsize=(14, 7), sharex=True)
        for k, (nm, i) in enumerate(pick.items()):
            ax[0, k].plot(tt_, Xt[i][0], lw=.6, color="k"); ax[0, k].set_title(f"{c} {nm}: test row {i}, signal EV {v[i]:.2f}", fontsize=8)
            ax[1, k].plot(tt_, Yt[i][0], lw=.7, label="truth"); ax[1, k].plot(tt_, pe[i][0], lw=.7, label="model")
            ax[2, k].plot(tt_, Xt[i][0] - Yt[i][0], lw=.7, label="truth"); ax[2, k].plot(tt_, pn[i][0], lw=.7, label="model")
        for r_, t_ in enumerate(("input (trace 1 of 3)", "signal", "noise")): ax[r_, 0].set_ylabel(t_)
        for a in ax[2]: a.set_xlabel("time in window (s)")
        ax[1, 0].legend(fontsize=7); plt.tight_layout(); save_fig(fig, f"fig_exploration_examples_{c}"); plt.show()
        del model, pe, pn, Xt, Yt
    if QUICK:
        Q = pd.DataFrame(QUICK); Q.to_csv(RESULTS/"table_exploration_quick_check.csv", index=False)
        print(Q[["dataset", "model_id", "median_ev_signal", "median_ev_noise", "test_loss", "mean_corr_signal", "runtime_min"]].round(4).to_string(index=False))
        log_entry("10 quick eval", "LSTM on own exploration test split + identity baseline", f"{len(runs)} datasets")
else:
    print("quick check skipped")

dataset          model_id  median_ev_signal  median_ev_noise  test_loss  mean_corr_signal  runtime_min
   VG2D  vg2d_LSTM_seed99            0.9410           0.4141     0.0903            0.8468      42.1850
   VG2D baseline_identity            0.9043          -9.4544     1.0010            0.7583          NaN
     F3    f3_LSTM_seed99            0.9692           0.6472     0.0645            0.8831      42.2312
     F3 baseline_identity            0.9035          -9.3678     1.0020            0.7508          NaN
  STR3D str3d_LSTM_seed99            0.9691           0.6772     0.0671            0.8693      42.2325
  STR3D baseline_identity            0.8993          -8.9338     1.0008            0.7482          NaN


## Next steps

1. Make the dataset **`wdn-runs-exploration`** from this version's output. It must contain `wdn_runs/Branch_Encoder_Decoder_LSTM_{vg2d,f3,str3d}/`; `results/`, `figures/` and `logs/` can stay in it.
2. Notebook 11 (`11_evaluate_exploration`) needs these attached: `wdn-exploration-mixtures`, `wdn-runs-exploration`, the earlier run dataset (`wdn-runs-all`), the STEAD and INSTANCE mixtures.
3. Add to the deviation log:
   - same script and hyperparameters as the INSTANCE run
   - 18,000 training rows per dataset, so fewer weight updates per epoch than the STEAD run (as for INSTANCE)
   - one seed